In [ ]:
import os
import polars as pl
import pandas as pd
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, HTML
from collections.abc import Iterator
from statsmodels.tsa.seasonal import STL, DecomposeResult
from prophet import Prophet

from src.etl import ETL
from src.model import ForecastModel

: 

In [ ]:
BASE_PATH: str = os.path.join('.', 'src', 'files')
RANDOM_SEED: float = 0

sns.set_theme(style="darkgrid", palette="dark")

In [ ]:
df_raw: pl.DataFrame = pl.read_csv(
    os.path.join(BASE_PATH, 'mock_data', 'mock_sales_data.csv'),
    schema_overrides = {
        'SaleDate': pl.Date,
        'ProductID': pl.String,
        'ASP': pl.Float32,
        'QT': pl.Int32,
    }
)

In [ ]:
df_raw.head(5)

In [ ]:
df_raw = df_raw.with_columns(
    pl.col('SaleDate').dt.year().alias('Y'),
    pl.col('SaleDate').dt.month().alias('M'),
    pl.concat_str(
        pl.col('SaleDate').dt.to_string('%Y'),
        pl.col('SaleDate').dt.to_string('%m'),
        separator = '/'
    ).alias('YearMonth')
)

In [ ]:
df_raw.describe()

In [ ]:
df_raw = df_raw.filter(
    (pl.col('QT') >= 0.0) & (pl.col('ASP') >= 0.0)
)

In [ ]:
df_raw.group_by(
    pl.col('ProductID')
).agg(
    pl.col('SaleDate').min().alias('Min. Date'),
    pl.col('SaleDate').max().alias('Max. Date'),
).sort(
    by='ProductID'
)

In [ ]:
df_raw = df_raw.filter(
    pl.col('SaleDate') < dt.date(2026, 6, 1)
)

In [ ]:
etl: ETL = ETL()

In [ ]:
df_QT_noiseless: pd.DataFrame = etl.prepare_dataframe(
    df = df_raw.to_pandas(),
    x = 'ProductID',
    y = 'QT'
)

df_QT_noiseless: pl.DataFrame = pl.from_pandas(df_QT_noiseless)

In [ ]:
df_QT_noiseless

In [ ]:
def line_plot(df: pl.DataFrame, x: str, y: str, hue: str | None = None) -> None:
    fig_in, ax_in = plt.subplots(figsize=(18, 6))
    sns.lineplot(data=df, x=x, y=y, hue=hue, ax=ax_in)

In [ ]:
line_plot(df_QT_noiseless, x='ds', y='y', hue='ProductID')

In [ ]:
noise: np.ndarray = np.ones(len(df_QT_noiseless))
products: list[str] = df_QT_noiseless['ProductID'].unique().sort().to_list()

for i, product in enumerate(products):
    mask: np.ndarray[bool] = (df_QT_noiseless["ProductID"] == product).to_numpy()
    mask_size: int = mask.sum()
    noise[mask]: Iterator[float] = np.random.default_rng(RANDOM_SEED + i).normal(1.0, 0.05, mask_size)

df_QT: pl.DataFrame = df_QT_noiseless.with_columns(
    (pl.col("y") * pl.Series("noise", noise)).round(2).alias("y")
)

In [ ]:
line_plot(df_QT, x='ds', y='y', hue='ProductID')

In [ ]:
df_QT: pd.DataFrame = df_QT.to_pandas()
df_QT['ds'] = pd.to_datetime(df_QT['ds'])

In [ ]:
def components_plot(df: pd.DataFrame, product: str) -> None:
    components_colors: dict[str, str] = {
        "Observed": "grey",
        "Trend": "blue",
        "Seasonality": "red",
        "Residual": "yellow",
    }

    fig, axes = plt.subplots(
        figsize = (22, 4),
        nrows = 1,
        ncols = 4,
    )

    fig.suptitle(f"STL Decomposition for QT. Product {product}", fontsize=16, fontweight="bold")

    product_series: pd.Series = (
        df[df["ProductID"] == product]
        .set_index("ds")["y"]
        .asfreq("MS")
    )
    stl: STL = STL(product_series, period=12, robust=True)
    result: DecomposeResult = stl.fit()

    components: dict[str, pd.Series] = {
        'Observed': product_series,
        'Trend': result.trend,
        'Seasonality': result.seasonal,
        'Residual': result.resid
    }

    for col_idx, (component_name, component_series) in enumerate(components.items()):
        axes[col_idx].set_title(
            component_name,
            fontsize = 14,
            fontweight = 'bold',
        )
        ax = axes[col_idx]

        if component_name in ('Seasonality', 'Residual'):
            ax.axhline(0, color='grey', linewidth=1, linestyle='--', alpha=0.6)

        if component_name == 'Residual':
            sns.scatterplot(
                x = component_series.index,
                y = component_series.values,
                ax = ax,
                color = components_colors[component_name],
            )
        else:
            sns.lineplot(
                x = component_series.index,
                y = component_series.values,
                ax = ax,
                color = components_colors[component_name],
            )

        ax.tick_params(axis="y", labelsize=9, left=True, labelleft=True)

        for spine in ax.spines.values():
                spine.set_linewidth(0.8)
                spine.set_color("grey")

    plt.tight_layout()
    plt.show()
    display(HTML("<div style='margin-bottom: 40px'></div>"))

In [ ]:
df_QT_P001: pd.DataFrame = df_QT[df_QT['ProductID'] == 'P001']
components_plot(df_QT_P001, 'P001')

In [ ]:
def seasonal_plot(df: pd.DataFrame, product: str) -> None:
    fig, axes = plt.subplots(figsize=(22, 4))
    fig.suptitle(f"Seasonal plot for QT. Product {product}", fontsize=16, fontweight="bold")

    sns.lineplot(
        data = df,
        x = 'month',
        y = 'y',
        hue = 'year',
        marker = 'o',
        legend = False
    )

    plt.tight_layout()

In [ ]:
def month_boxplot(df: pd.DataFrame, product: str) -> None:
    fig, axes = plt.subplots(figsize=(22, 4))
    fig.suptitle(f"Month boxplot for QT. Product {product}", fontsize=16, fontweight="bold")

    sns.boxplot(
        data = df,
        x = 'month',
        y = 'y'
    )

    plt.tight_layout()

In [ ]:
df_QT_P001_seasonal: pd.DataFrame = df_QT_P001.copy()
df_QT_P001_seasonal['month'] = df_QT_P001_seasonal['ds'].dt.month
df_QT_P001_seasonal['year'] = df_QT_P001_seasonal['ds'].dt.year

df_P001_QT_seasonal: pd.DataFrame = df_QT_P001_seasonal[df_QT_P001_seasonal['ProductID'] == 'P001']
seasonal_plot(df_QT_P001_seasonal, 'P001')

In [ ]:
month_boxplot(df_QT_P001_seasonal, 'P001')

In [ ]:
prophet_anomaly_detector: Prophet = Prophet(interval_width=0.99)
prophet_anomaly_detector.fit(df_QT_P001[['ds', 'y']])
outlier_detection_QT_P001: pd.DataFrame = prophet_anomaly_detector.predict(df_QT_P001)

df_QT_P001 = df_QT_P001.merge(
    outlier_detection_QT_P001[['ds', 'yhat_upper', 'yhat_lower']],
    on='ds',
    how='left'
)

df_QT_P001['outlier']: pd.Series = np.where(
    (df_QT_P001['yhat_lower'] > df_QT_P001['y']) |
    (df_QT_P001['yhat_upper'] < df_QT_P001['y']),
    True, False
)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
sns.lineplot(
    x = df_QT_P001['ds'],
    y = df_QT_P001['y'],
    ax = ax,
    linewidth = 1,
    label = 'Actual'
)

sns.scatterplot(
    x = df_QT_P001[df_QT_P001['outlier']]['ds'],
    y = df_QT_P001.loc[df_QT_P001['outlier'], 'y'],
    ax = ax,
    color = 'red',
    s = 60,
    zorder = 5,
    label = 'Outlier'
)

ax.set_title('QT detected outliers for Product P001')
plt.tight_layout()
plt.show()

In [ ]:
df_QT_P001[df_QT_P001['outlier'] == True]

In [ ]:
df_QT_P001_clean: pd.DataFrame = df_QT_P001.copy()
df_QT_P001_clean.loc[df_QT_P001_clean['outlier'], 'y'] = np.nan

In [ ]:
line_plot(df_QT_P001_clean, x='ds', y='y')

In [ ]:
forecast_model: ForecastModel = ForecastModel(
    material = 'P001',
    target = 'QT',
    model_name = 'mock_data_model'
)

hyperparameters_QT_P001: dict = forecast_model.hyperparameterize(
    df_QT_P001_clean,
    n_trials = 10,
    store = False
)